# Create Delta Table

### **Using SQL API**

In [0]:
%sql
SHOW CATALOGS

catalog
dbacademy
delta
deltalakecatalog1
samples
system
workspace


In [0]:
%sql
DROP CATALOG IF EXISTS deltalakecatalog1 CASCADE;
CREATE CATALOG deltalakecatalog1;

In [0]:
%sql
create table if not exists deltalakecatalog1.default.first_table 
(id INT, salary INT)

###  Using Delta / Python API 

In [0]:
from delta.tables import DeltaTable

In [0]:
# Create table in the metastore
DeltaTable.createIfNotExists(spark) \
  .tableName("deltalakecatalog1.default.people10m") \
  .addColumn("id", "INT") \
  .addColumn("salary", "INT") \
  .execute()

## Constraints

In [0]:
%sql
create table if not exists deltalakecatalog1.default.first_table 
(id INT NOT NULL, salary INT UNIQUE)

In [0]:
%sql
drop table deltalakecatalog1.default.first_table;

## Generated Columns

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql.types import LongType, IntegerType, StringType
from delta.tables import IdentityGenerator

DeltaTable.create(spark) \
  .tableName("deltalakecatalog1.default.first_table") \
  .addColumn("id_col1", dataType=LongType(), generatedAlwaysAs=IdentityGenerator()) \
  .addColumn("salary", dataType=IntegerType()) \
  .addColumn("name", dataType=StringType()) \
  .execute()

## Computed Columns

In [0]:
DeltaTable.create(spark) \
  .tableName("deltalakecatalog1.default.second_table") \
  .addColumn("salaryAfterTaxes", dataType=LongType(), generatedAlwaysAs="CAST((salary * 0.85) as BIGINT)") \
  .addColumn("salary", dataType=IntegerType()) \
  .addColumn("name", dataType=StringType()) \
  .execute()

In [0]:
%sql
insert into deltalakecatalog1.default.second_table (salary,name)
values(90000,'John Doe'),
(80000,'Jane Doe');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
select * from deltalakecatalog1.default.second_table;

salaryAfterTaxes,salary,name
76500,90000,John Doe
68000,80000,Jane Doe


## CH2 - Delta Log

### Creating Dataframe

In [0]:
data=[(1,100,'a'),(2,200,'b'),(3,300,'c'),(4,400,'d')]
df=spark.createDataFrame(data,["id_col1","salary","name"])

df.display()

id_col1,salary,name
1,100,a
2,200,b
3,300,c
4,400,d


### Write dataframe data to volumes as delta files

In [0]:
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/deltalakecatalog1/default/deltavol/demosink/")

re-run write command again

In [0]:
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/deltalakecatalog1/default/deltavol/demosink/")

In [0]:
df=spark.read.format("json")\
.load("/Volumes/deltalakecatalog1/default/deltavol/demosink/_delta_log/00000000000000000001.json")

display(df)

add,commitInfo
null,"List(0730-074520-x1vcpmlj-v2n, Databricks-Runtime/18.x-aarch64-photon-scala2.13, true, WriteSerializable, List(2547137114558828), WRITE, List(1, 1294, 4), List(Append, [], false), 1acfe5f4-67f4-45ba-ba66-a4e5abb39700, 0, List(true, false), 1785398306056, 0e2f65a0-57d1-4041-8167-4dea1aebc9cf, 76548768579520, aishwaryasr.reddy@gmail.com)"
"List(true, 1785398306000, part-00000-37dcd256-c23a-4544-9165-2c4cef08e802.c000.snappy.parquet, 1294, {""numRecords"":4,""minValues"":{""id_col1"":1,""salary"":100,""name"":""a""},""maxValues"":{""id_col1"":4,""salary"":400,""name"":""d""},""nullCount"":{""id_col1"":0,""salary"":0,""name"":0},""tightBounds"":true}, List(1785398306000000, 1785398306000000, 1785398306000000, 268435456))",null


## Ch3 - Delta Lake Schema Enforcement & Evolution

In [0]:
data=[(1,100,'a',1),(2,200,'b',1),(3,300,'c',1),(4,400,'d',1)]
df=spark.createDataFrame(data,["id_col1","salary","name","suspicious_col"])

df.display()

id_col1,salary,name,suspicious_col
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1


**Schema Enforcement**

In [0]:
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/deltalakecatalog1/default/deltavol/demosink/")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7759114376387705>, line 3
      1 df.write.format("delta")\
      2     .mode("append")\
----> 3     .save("/Volumes/deltalakecatalog1/default/deltavol/demosink/")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._set_command_in_p

**Schema Evolution**

In [0]:
df.write.format("delta")\
    .mode("append")\
    .option("mergeSchema","true")\
    .save("/Volumes/deltalakecatalog1/default/deltavol/demosink/")

### **Read Delta Data**

#### **Table**

In [0]:
%sql
select * from deltalakecatalog1.default.second_table;

salaryAfterTaxes,salary,name
76500,90000,John Doe
68000,80000,Jane Doe


#### **Volumes**

In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/demosink/`;

id_col1,salary,name,suspicious_col
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1
1,100,a,null
2,200,b,null
3,300,c,null
4,400,d,null
1,100,a,null
2,200,b,null


### **Schema Overwrite**

In [0]:
data=[(1,100,'a',1),(2,200,'b',1),(3,300,'c',1),(4,400,'d',1)]
df=spark.createDataFrame(data,["cust_id","income","name","tip"])

df.display()

cust_id,income,name,tip
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save('/Volumes/deltalakecatalog1/default/deltavol/demosink/')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7759114376387716>, line 3
      1 df.write.format("delta")\
      2     .mode("overwrite")\
----> 3     .save('/Volumes/deltalakecatalog1/default/deltavol/demosink/')

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._set_command_i

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .save('/Volumes/deltalakecatalog1/default/deltavol/demosink/')

In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/demosink/`;

id_col1,salary,name,suspicious_col,cust_id,income,tip
null,null,a,null,1,100,1
null,null,b,null,2,200,1
null,null,c,null,3,300,1
null,null,d,null,4,400,1


overwrite

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema","true")\
    .save('/Volumes/deltalakecatalog1/default/deltavol/demosink/')

In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/demosink/`;

cust_id,income,name,tip
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1


In [0]:
df=spark.read.format("json")\
.load("/Volumes/deltalakecatalog1/default/deltavol/demosink/_delta_log/00000000000000000004.json")

display(df)

add,commitInfo,metaData,remove
null,"List(0730-144534-wgecx8cs-v2n, Databricks-Runtime/18.x-aarch64-photon-scala2.13, false, WriteSerializable, List(2547137114558828), WRITE, List(0, 1, 1520, 4, 1520, 1), List(true, Overwrite, [], false), 7195ef35-99bc-4bcd-82ee-decfe26dec0f, 3, List(true, false), 1785425274961, d27a526e-3861-4f03-a923-7e46acc20210, 76548768579520, aishwaryasr.reddy@gmail.com)",null,null
null,null,"List(List(true), 1785398166793, List(parquet), c6ec380c-15cd-454b-897f-a161a8cd8c54, List(), {""type"":""struct"",""fields"":[{""name"":""cust_id"",""type"":""long"",""nullable"":true,""metadata"":{}},{""name"":""income"",""type"":""long"",""nullable"":true,""metadata"":{}},{""name"":""name"",""type"":""string"",""nullable"":true,""metadata"":{}},{""name"":""tip"",""type"":""long"",""nullable"":true,""metadata"":{}}]})",null
"List(true, 1785425275000, part-00000-25783110-b644-4d76-8c9e-a074e79c8ca1.c000.snappy.parquet, 1520, {""numRecords"":4,""minValues"":{""cust_id"":1,""income"":100,""name"":""a"",""tip"":1},""maxValues"":{""cust_id"":4,""income"":400,""name"":""d"",""tip"":1},""nullCount"":{""cust_id"":0,""income"":0,""name"":0,""tip"":0},""tightBounds"":true}, List(1785425275000000, 1785425275000000, 1785425275000000, 268435456))",null,null,null
null,null,null,"List(true, 1785425274955, true, part-00000-9c52c30a-73d2-41bc-a276-b648678845bd.c000.snappy.parquet, 1520, List(1785425133000000, 1785425133000000, 1785425133000000, 268435456))"
